# Fracture Detection — End-to-End Training (Local / Colab)

**Medical Image Analysis for Bone Fracture Detection Using Deep Learning**

This notebook trains the IEEE Access 2025 classification stack and the YOLOv8 detector from the project abstract on datasets from ~1k to **20,000+** radiographs.

| Model | Role | Reported metric |
| --- | --- | --- |
| YOLOv8 | Real-time localization | mAP50 = 0.86 (3,316 / 399 split) |
| VGG-16 + Softmax | Primary classifier | accuracy 0.95 |
| VGG-16 + Random Forest | Best hybrid ensemble | accuracy 0.95 |
| ResNet-50 + SVM | Comparative hybrid | ~0.93 |
| EfficientNetB0 + XGBoost | Comparative hybrid | ~0.41 |

**Classes (morphology):** Avulsion, Comminuted, Fracture-Dislocation, Greenstick, Hairline, Impacted, Longitudinal, Oblique, Pathological, Spiral.

**Classes (anatomy):** Elbow, Fingers, Forearm, Humerus, Humerus Fracture, Shoulder, Wrist, Pelvis.


In [ ]:
# Runtime: Google Colab GPU (T4/A100) or local CUDA workstation
import os, sys, pathlib
IN_COLAB = "google.colab" in sys.modules
print("Colab:", IN_COLAB)
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = pathlib.Path("/content/bone_fracture_system")
else:
    ROOT = pathlib.Path(".").resolve()
    if (ROOT / "models").exists() is False and (ROOT / "bone_fracture_system").exists():
        ROOT = ROOT / "bone_fracture_system"
print("ROOT", ROOT)


In [ ]:
# Dependencies (skip packages already present)
%pip install -q ultralytics scikit-learn xgboost opencv-python-headless pydicom joblib pandas matplotlib
# TensorFlow and PyTorch: use Colab defaults or your CUDA wheels locally.


## 1. Dataset layout

Classification (ImageFolder, 256×256 as in Torne et al.):

```
data/classification/train/<class_name>/*.png
data/classification/val/<class_name>/*.png
```

Detection (Ultralytics YOLO):

```
data/yolo/images/train  +  data/yolo/labels/train
data/yolo/images/val    +  data/yolo/labels/val
```

On Colab, point `DATA_ROOT` at a Drive folder or Kaggle download (bone fracture X-ray collections). Scale by mixing public sets until you exceed 20k labeled images; keep a held-out validation split (~0.2).


In [ ]:
from pathlib import Path
DATA_ROOT = ROOT / "data"
CLS_TRAIN = DATA_ROOT / "classification" / "train"
CLS_VAL = DATA_ROOT / "classification" / "val"
YOLO_YAML = DATA_ROOT / "yolo" / "data.yaml"
WEIGHTS = ROOT / "weights"
WEIGHTS.mkdir(parents=True, exist_ok=True)
print("classification train exists:", CLS_TRAIN.exists())
print("yolo yaml exists:", YOLO_YAML.exists())


## 2. VGG-16 Softmax (paper architecture)

Frozen ImageNet VGG-16, flatten, Dense-ReLU, Dropout, Dense-ReLU, Softmax. Adam `lr=5e-4`, categorical cross-entropy, 20 epochs, batch 32.


In [ ]:
import sys
sys.path.insert(0, str(ROOT))
from models.train_vgg16_rf import train_softmax, train_random_forest

if CLS_TRAIN.exists():
    train_softmax(CLS_TRAIN, CLS_VAL, epochs=20, output=WEIGHTS / "vgg16_softmax.h5")
else:
    print("Skip softmax: classification folders not found.")


## 3. VGG-16 + Random Forest (top hybrid in the paper)

In [ ]:
if CLS_TRAIN.exists():
    train_random_forest(CLS_TRAIN, CLS_VAL, WEIGHTS / "vgg16_random_forest.joblib")
else:
    print("Skip RF: classification folders not found.")


## 4. Comparative ensembles — ResNet-50 + SVM and EfficientNetB0 + XGBoost

In [ ]:
from models.train_ensembles import train_resnet_svm, train_efficientnet_xgb
if CLS_TRAIN.exists():
    train_resnet_svm(CLS_TRAIN, CLS_VAL, WEIGHTS / "resnet50_svm.joblib")
    train_efficientnet_xgb(CLS_TRAIN, CLS_VAL, WEIGHTS / "efficientnetb0_xgboost.joblib")
else:
    print("Skip comparative ensembles.")


## 5. YOLOv8 multi-class detector

Abstract protocol used 3,316 train / 399 val images and seven anatomical labels (extended here to 18 names in `data/yolo/data.yaml`). Start from `yolov8n.pt` or `yolov8s.pt`.


In [ ]:
from models.train_yolov8 import train as train_yolo, ensure_yaml
yaml_path = ensure_yaml(YOLO_YAML)
images_train = DATA_ROOT / "yolo" / "images" / "train"
if images_train.exists() and any(images_train.iterdir()):
    train_yolo(yaml_path, epochs=80, imgsz=640, model_name="yolov8n.pt", project=ROOT / "runs" / "detect", batch=8)
else:
    print("Skip YOLO: no labeled detection images yet. Add data under", images_train)


## 6. Grad-CAM sanity check on a sample radiograph

In [ ]:
import cv2
from models.gradcam import generate_gradcam
sample = None
for folder in (CLS_VAL, CLS_TRAIN, ROOT / "static" / "uploads"):
    if folder.exists():
        hits = list(folder.rglob("*.png")) + list(folder.rglob("*.jpg"))
        if hits:
            sample = hits[0]
            break
if sample:
    img = cv2.imread(str(sample))
    cam = generate_gradcam(img)
    out = WEIGHTS / "gradcam_preview.png"
    cv2.imwrite(str(out), cam)
    print("Wrote", out)
else:
    print("No sample image found for Grad-CAM preview.")


## 7. Export weights into the Flask app

Copy `weights/*.h5`, `*.joblib`, and `yolov8_fracture.pt` next to the Flask process, then restart `python app.py`. Demo mode turns off automatically when those files are present.

**Clinical reminder:** research software only — not a medical device.
